1.Diseño e implementación del Agente 1:Agente Reactivo Simple Termostato de Centro de Datos.1. Ficha PEASPerformance (Medida de desempeño): Mantener la temperatura de la sala en el rango óptimo ($18^\circ\text{C} - 22^\circ\text{C}$), minimizando el consumo de energía y evitando sobrecalentamientos por encima de $30^\circ\text{C}$.Environment (Entorno): Sala de servidores equipada con sensores térmicos y un sistema HVAC de dos etapas.Actuators (Acciones): APAGAR_ENFRIAMIENTO, ENFRIAMIENTO_MODERADO, ENFRIAMIENTO_MAXIMO, ALERTA_EMERGENCIA_Y_MAXIMO.Sensors (Percepciones): Lectura continua de temperatura en grados Celsius ($T$).2. Reglas de decisión (Condición-Acción)$$\text{Acción} = \begin{cases}  \text{APAGAR\_ENFRIAMIENTO} & \text{si } T < 18^\circ\text{C} \\ \text{MANTENER\_ESTADO} & \text{si } 18^\circ\text{C} \le T \le 22^\circ\text{C} \\ \text{ENFRIAMIENTO\_MODERADO} & \text{si } 22^\circ\text{C} < T \le 26^\circ\text{C} \\ \text{ENFRIAMIENTO\_MAXIMO} & \text{si } 26^\circ\text{C} < T \le 30^\circ\text{C} \\ \text{ALERTA\_EMERGENCIA\_Y\_MAXIMO} & \text{si } T > 30^\circ\text{C} \end{cases}$$

In [1]:
class AgenteTermostatoReactivo:
    """Agente reactivo simple basado en reglas condición-acción sobre lecturas puntuales."""
    
    def decidir_accion(self, temperatura: float) -> str:
        if temperatura < 18.0:
            return "APAGAR_ENFRIAMIENTO"
        elif 18.0 <= temperatura <= 22.0:
            return "MANTENER_ESTADO"
        elif 22.0 < temperatura <= 26.0:
            return "ENFRIAMIENTO_MODERADO"
        elif 26.0 < temperatura <= 30.0:
            return "ENFRIAMIENTO_MAXIMO"
        else:
            return "ALERTA_EMERGENCIA_Y_MAXIMO"

# Entorno de pruebas
class EntornoCentroDatos:
    def __init__(self, temp_inicial: float):
        self.temperatura = temp_inicial

    def actualizar_entorno(self, accion: str):
        if accion == "APAGAR_ENFRIAMIENTO":
            self.temperatura += 1.5
        elif accion == "ENFRIAMIENTO_MODERADO":
            self.temperatura -= 1.0
        elif accion in ["ENFRIAMIENTO_MAXIMO", "ALERTA_EMERGENCIA_Y_MAXIMO"]:
            self.temperatura -= 2.5
        elif accion == "MANTENER_ESTADO":
            self.temperatura += 0.2  # Calentamiento natural por cómputo

In [2]:
def simular_termostato():
    entorno = EntornoCentroDatos(temp_inicial=29.0)
    agente = AgenteTermostatoReactivo()
    
    print("--- SIMULACIÓN AGENTE 1: TERMOSTATO REACTIVO ---")
    for t in range(1, 6):
        temp_actual = entorno.temperatura
        accion = agente.decidir_accion(temp_actual)
        print(f"Paso {t} | Temp. percibida: {temp_actual:.1f}°C | Acción elegida: {accion}")
        entorno.actualizar_entorno(accion)

simular_termostato()

--- SIMULACIÓN AGENTE 1: TERMOSTATO REACTIVO ---
Paso 1 | Temp. percibida: 29.0°C | Acción elegida: ENFRIAMIENTO_MAXIMO
Paso 2 | Temp. percibida: 26.5°C | Acción elegida: ENFRIAMIENTO_MAXIMO
Paso 3 | Temp. percibida: 24.0°C | Acción elegida: ENFRIAMIENTO_MODERADO
Paso 4 | Temp. percibida: 23.0°C | Acción elegida: ENFRIAMIENTO_MODERADO
Paso 5 | Temp. percibida: 22.0°C | Acción elegida: MANTENER_ESTADO


# Agente 2: Agente Limpiador de Cuadrícula

Tipo: Basado en Modelos (Estado Interno)

1. Ficha PEAS
Componente	Descripción
P (Performance)	Maximizar el número de celdas limpias, minimizar el número de movimientos en vano y evitar colisiones contra muros.
E (Environment)	Cuadrícula de 2 × 2 (A, B, C, D) con presencia de suciedad aleatoria y paredes. Entorno parcialmente observable (el agente solo percibe la celda donde está ubicado).
A (Actuators)	Limpiar, Mover_Izquierda, Mover_Derecha, Mover_Arriba, Mover_Abajo, Permanecer.
S (Sensors)	Sensor de ubicación actual y sensor de estado local (Limpio / Sucio).
2. Reglas de decisión
Actualización de modelo: registrar en la memoria interna el estado de la celda actual.
Si Estado(Celda_Actual) == Sucio → Acción: LIMPIAR.
Si Estado(Celda_Actual) == Limpio:
Consultar el mapa interno.
Seleccionar la celda adyacente que sea Desconocida o figure como Sucia.
Si todas las celdas adyacentes están Limpias en el modelo → Acción: PERMANECER.

In [3]:
from typing import Dict, Tuple

class AgenteLimpiadorConModelo:
    """Agente basado en modelos que mantiene un mapa del entorno para tomar decisiones."""
    
    def __init__(self, posicion_inicial: Tuple[int, int]):
        self.posicion = posicion_inicial
        # Modelo interno del entorno: (x, y) -> estado conocido
        self.modelo_mundo: Dict[Tuple[int, int], str] = {
            (0, 0): "desconocido", (0, 1): "desconocido",
            (1, 0): "desconocido", (1, 1): "desconocido"
        }

    def decidir_accion(self, percibir_suciedad: bool) -> str:
        # Actualizar estado interno
        self.modelo_mundo[self.posicion] = "sucio" if percibir_suciedad else "limpio"
        
        # Regla 1: Limpiar si está sucio
        if percibir_suciedad:
            return "LIMPIAR"
            
        # Regla 2: Navegar a celda no explorada o sucia conocida
        vecinos = self._obtener_vecinos(self.posicion)
        for pos_vecina in vecinos:
            if self.modelo_mundo.get(pos_vecina) in ["desconocido", "sucio"]:
                self.posicion = pos_vecina
                return f"MOVER_A_{pos_vecina}"
                
        return "PERMANECER"

    def _obtener_vecinos(self, pos: Tuple[int, int]):
        x, y = pos
        candidatos = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
        return [p for p in candidatos if p in self.modelo_mundo]

class EntornoGrid:
    def __init__(self):
        self.estado_real = {
            (0, 0): True,   # Sucio
            (0, 1): False,  # Limpio
            (1, 0): True,   # Sucio
            (1, 1): False   # Limpio
        }

    def es_sucio(self, pos: Tuple[int, int]) -> bool:
        return self.estado_real.get(pos, False)

    def limpiar_posicion(self, pos: Tuple[int, int]):
        self.estado_real[pos] = False

In [4]:
def simular_limpiador():
    entorno = EntornoGrid()
    pos_inicial = (0, 0)
    agente = AgenteLimpiadorConModelo(posicion_inicial=pos_inicial)
    
    print("\n--- SIMULACIÓN AGENTE 2: LIMPIADOR CON MODELO INTERNO ---")
    pos_actual = pos_inicial
    for t in range(1, 6):
        sucio = entorno.es_sucio(pos_actual)
        accion = agente.decidir_accion(percibir_suciedad=sucio)
        
        print(f"Paso {t} | Pos: {pos_actual} | Percibe Sucio: {sucio} | Acción: {accion}")
        
        if accion == "LIMPIAR":
            entorno.limpiar_posicion(pos_actual)
        elif accion.startswith("MOVER_A_"):
            # Extraer nueva posición del texto
            pos_actual = agente.posicion

simular_limpiador()


--- SIMULACIÓN AGENTE 2: LIMPIADOR CON MODELO INTERNO ---
Paso 1 | Pos: (0, 0) | Percibe Sucio: True | Acción: LIMPIAR
Paso 2 | Pos: (0, 0) | Percibe Sucio: False | Acción: MOVER_A_(1, 0)
Paso 3 | Pos: (1, 0) | Percibe Sucio: True | Acción: LIMPIAR
Paso 4 | Pos: (1, 0) | Percibe Sucio: False | Acción: MOVER_A_(1, 1)
Paso 5 | Pos: (1, 1) | Percibe Sucio: False | Acción: MOVER_A_(0, 1)
